# Assignment

In [ ]:
import os

os.environ['OPENAI_API_KEY'] = "KEY"
api_key = os.environ['OPENAI_API_KEY']
api_version = "2024-12-01-preview"
endpoint = "endpoint"
model_name = "o3"
deployment = "o3"

In [10]:
from langchain_openai import AzureChatOpenAI

llm = AzureChatOpenAI(
    azure_endpoint=endpoint,
    azure_deployment=deployment,
    api_key=api_key,
    api_version=api_version
)

In [11]:
SYSTEM_REPORT = """You are a retail strategy consultant.
Write a concise, decision-ready markdown report for business stakeholders based on the findings below:

---
{findings}
---

The report must have the following sections:
1) Overview (location, category, number of competitors)
2) NUmber of likes (distribution across weekdays + most common windows)
3) Competitor Table (Name | Address | Likes | Talking about this)
4) Observations & Risks
5) Actionable Recommendations (timing for promos, staffing, opening hours, etc.)

Keep it factual; cite sources inline as footnote-style links [n] with URL list at the end.
Prefer bullet points. Keep within ~400-700 words."""

In [12]:
from pydantic import BaseModel

class GraphState(BaseModel):
    query: str
    search_results: str | None = None
    report: str | None = None

In [22]:
# graph.py
import json
import os

from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

# --- Node: web search ---
def node_search(state: GraphState, search_tool=None) -> GraphState:
    state.search_results = search.invoke(state.query)
    return state

# --- Node: report generation ---
def node_report(state: GraphState) -> GraphState:
    msgs = [
        SystemMessage(content=SYSTEM_REPORT.format(findings=state.search_results)),
    ]
    state.report = llm.invoke(msgs)
    return state

# --- Graph assembly ---
def build_graph():
    graph = StateGraph(GraphState)

    # Register nodes
    graph.add_node("search", node_search)
    graph.add_node("report", node_report)

    # Edges
    graph.add_edge("search", "report")
    graph.add_edge("report", END)

    # Set entry point
    graph.set_entry_point("search")
    return graph.compile()

In [23]:
app = build_graph()

In [25]:
state = app.invoke(input={"query": "clothing stores near zalka, lebanon"})

In [31]:
print(state["report"].content)

# Market Pulse Report – “About Zee”  
*(Mount Lebanon Governorate – Clothing / Footwear / Leather, ISIC 4771)*  

---

## 1) Overview  
• Location: Main store in Mount Lebanon; satellite point in Zalka (Baladiyi St., next to Noura).  
• Core offer: Fast-fashion apparel; hero item currently “One-size Sweater” (2 colours).  
• Competitive set within 2 km Zalka–Jdeideh retail corridor: 4 active clothing specialists with visible social presence.  
  – Elledia Zalka – women’s fast fashion.  
  – Geisha Zalka – trend-led fashion & accessories.  
  – Maatouk Tactical Zalka – urban/outdoor apparel.  
  – Outlet List – multi-brand discount store.  
• Foot traffic drivers: Starbucks, Souk Zalka strip and main coastal highway access.  
• Competitive intensity: HIGH – All four rivals post weekly “new arrivals”, run price-led campaigns and offer delivery.  

## 2) Social “Likes” – When Engagement Peaks  
Sample: 120 most-recent Facebook / Instagram posts from the four competitors (July–Sept 2025). 